# Python Annotations — A Primer

This short notebook is a **prerequisite** for [3.graph_with_reducers.ipynb](3.graph_with_reducers.ipynb).

That notebook defines graph state like this:

```python
class ListState(TypedDict):
    items: Annotated[list[str], operator.add]
```

To read that comfortably you need to understand three things:

1. **Type annotations** — how Python lets you *label* the expected type of a variable, argument, or return value.
2. **`TypedDict`** — a dictionary whose keys have declared types.
3. **`Annotated`** — a way to attach *extra metadata* to a type (LangGraph uses this to attach a **reducer**).

Let's build up to it step by step.

## 1. What is a type annotation?

An annotation is a `: type` label. It documents *intent* — what kind of value is expected.

You can annotate variables, function parameters, and return values.

### Is this like type declarations in other languages?

Syntactically, yes — `name: str` mirrors `String name` (Java/C#), `let name: string` (TypeScript), or `var name string` (Go). But there is one crucial difference: **enforcement**.

| Language | Declaration | Checked when? |
|----------|-------------|---------------|
| Java / C# / C++ / Rust / Go | `String name = ...` | **Compiler** rejects type mismatches *before* the program runs |
| TypeScript | `let name: string = ...` | A separate compiler checks, then types are **erased** before execution |
| **Python** | `name: str = ...` | **Not checked at runtime** — optional tools (Pyright, mypy) or your editor check |

So Python annotations behave most like **TypeScript**: hints validated by a separate, *optional* checker and ignored by the interpreter itself. Nothing stops `count: int = "oops"` from running — we'll demonstrate this in the next section.

This *lack* of runtime enforcement is exactly what lets libraries repurpose annotations to carry extra meaning — the trick LangGraph uses with `Annotated` later in this notebook. (The closest analogues elsewhere are Java annotations like `@NotNull` or C# attributes like `[Required]`, which also attach metadata read via reflection.)


In [2]:
# Variable annotations
name: str = "Ada"
age: int = 36
scores: list[int] = [90, 85, 100]


# Function annotations: each parameter has a type, and '-> bool' is the return type
def is_passing(score: int, threshold: int = 60) -> bool:
    return score >= threshold


print(name, age, scores)
print("is_passing(85):", is_passing(85))

Ada 36 [90, 85, 100]
is_passing(85): True


## 2. Annotations are *not* enforced at runtime

This is the single most important thing to understand.

Python **does not check** annotations when your code runs. They are hints — for humans, for editors (autocomplete), and for static type checkers. In **VS Code**, that checker is **Pylance**, which is built on **Pyright** and performs the same kind of type checking as the standalone `pyright` and `mypy` tools. You can turn it up or down with the `python.analysis.typeCheckingMode` setting (`off` / `basic` / `standard` / `strict`).

Even so, the checker only flags issues in your editor — the Python interpreter still runs the code regardless. You can assign the "wrong" type and Python won't complain at runtime.


In [3]:
count: int = "this is not an int"  # no error at runtime!
print(count)
print("type is actually:", type(count).__name__)

this is not an int
type is actually: str


Because they aren't enforced, libraries are free to *read* annotations and give them meaning. LangGraph does exactly this: it inspects your state annotations to discover reducers. We'll see how below.

In [4]:
# Annotations are stored in the __annotations__ dictionary and can be inspected
def greet(person: str, times: int = 1) -> str:
    return ("Hi " + person + "! ") * times


print(greet.__annotations__)

{'person': <class 'str'>, 'times': <class 'int'>, 'return': <class 'str'>}


## 3. Common typing constructs

Modern Python (3.9+) lets you write container types directly:

| Annotation | Meaning |
|------------|---------|
| `list[str]` | a list of strings |
| `dict[str, int]` | a dict mapping str keys to int values |
| `tuple[int, int]` | a 2-tuple of ints |
| `str \| None` | a string **or** `None` (optional) |

The `str | None` form (Python 3.10+) is equivalent to the older `Optional[str]`.

In [5]:
def first_or_none(items: list[str]) -> str | None:
    return items[0] if items else None


print(first_or_none(["apple", "banana"]))
print(first_or_none([]))

apple
None


## 4. `TypedDict` — a dict with typed keys

A `TypedDict` looks like a class, but instances are plain dictionaries. It declares which keys exist and what type each value should be.

This is what LangGraph uses to describe **graph state**.

In [6]:
from typing_extensions import TypedDict

class Person(TypedDict):
    name: str
    age: int


# An instance is just a regular dict
p: Person = {"name": "Ada", "age": 36}
print(p)
print("It's really a dict:", isinstance(p, dict))
print("Declared keys/types:", Person.__annotations__)

{'name': 'Ada', 'age': 36}
It's really a dict: True
Declared keys/types: {'name': <class 'str'>, 'age': <class 'int'>}


## 5. `Annotated` — attach metadata to a type

`Annotated[T, *metadata]` means "the type is `T`, **plus** some extra information".

- The **first** argument is the real type (e.g. `list[str]`).
- Everything **after** it is metadata that Python ignores but libraries can read.

```python
Annotated[list[str], "some metadata"]
#          ^^^^^^^^^  ^^^^^^^^^^^^^^^
#          the type   the metadata
```

In [7]:
from typing import Annotated, get_args, get_type_hints

# Attach a human-readable note as metadata
Age = Annotated[int, "years since birth"]

# get_args splits it into (type, *metadata)
print("args:", get_args(Age))

# The underlying type is still just int
print("__metadata__:", Age.__metadata__)

args: (<class 'int'>, 'years since birth')
__metadata__: ('years since birth',)


### The metadata can be *anything* — including a function

This is the key insight. Because metadata is arbitrary, a library can put a **function** there and later call it. LangGraph puts a **reducer function** in the metadata of each state key.

In [8]:
import operator

# here we are calling operator.add (note the parentheses + args) to show its functionality
print(operator.add([1, 2], [3]))

# We hand the function OBJECT itself to Annotated as metadata. 
# Nothing is invoked — the reducer (langgraph concept for this, to be explained later) just sits there,
# waiting to be called later (that happens below in section 6).
AccumulatingList = Annotated[list[str], operator.add]

the_type, the_reducer = get_args(AccumulatingList)
print("type   :", the_type)
print("reducer:", the_reducer)            # <function add> — the function, not a result
print("reducer is operator.add:", the_reducer is operator.add)


[1, 2, 3]
type   : list[str]
reducer: <built-in function add>
reducer is operator.add: True


## 6. Putting it together the way LangGraph does

A LangGraph state is a `TypedDict` where a key can be annotated with a reducer:

```python
class ListState(TypedDict):
    items: Annotated[list[str], operator.add]
```

Reading that out loud:

> *"`items` is a `list[str]`, and when nodes update it, merge the updates with `operator.add` (concatenation) instead of overwriting."*

Below we **simulate** what LangGraph does: read the reducer out of the annotation and use it to merge two updates.

In [9]:
from typing_extensions import TypedDict 

class ListState(TypedDict):
    items: Annotated[list[str], operator.add]


# 1. Discover the reducer attached to the 'items' key
hints = get_type_hints(ListState, include_extras=True)  # include_extras keeps Annotated metadata
items_type, items_reducer = get_args(hints["items"])
print("reducer for 'items':", items_reducer.__name__)

reducer for 'items': add


In [ ]:

# 2. Simulate two nodes each returning a partial update
state: ListState = {"items": []}
update_from_node_a = ["apple", "banana"]
update_from_node_b = ["carrot", "spinach"]

# 3. Apply the reducer to merge, instead of overwriting
state["items"] = items_reducer(state["items"], update_from_node_a)
state["items"] = items_reducer(state["items"], update_from_node_b)
print("merged items:", state["items"])

Compare that to a key **without** a reducer, where each update simply replaces the previous value:

In [ ]:
value = 0
value = 1    # node A overwrites
value = 11   # node B overwrites
print("final value (overwrite, no reducer):", value)

## Summary

- **Annotations** (`x: int`) label expected types and are **not enforced** at runtime.
- **`TypedDict`** describes a dict with typed keys — LangGraph uses it for **graph state**.
- **`Annotated[T, meta]`** attaches metadata to a type; the metadata can be a **function**.
- **LangGraph reads a reducer function** out of the `Annotated` metadata to decide how to merge state updates (append/sum/etc.) instead of overwriting.

You're now ready for the real thing 👉 [3.graph_with_reducers.ipynb](3.graph_with_reducers.ipynb), where `Annotated[list[str], operator.add]` and the built-in `add_messages` reducer drive how graph state evolves.